In [1]:
def build_model2(input_lenght=4, depth=2, n_class=2):

    model = Sequential()

    model.add(Dense(
        4,
        input_shape=(input_lenght,),
        activation='relu'
    ))

    for _ in range(depth - 1):

        model.add(Dense(
            4,
            activation='relu',
            use_bias=False
        ))

    model.add(Dense(
        n_class,
        activation='softmax',
        use_bias=False
    ))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model = build_model2(depth=3, n_class=2)
fit = model.fit(X_train_sv, y_train_sv,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                validation_data=(X_test_sv, y_test_sv),
                verbose=False,
                shuffle=True)    

In [ ]:
import numpy as np
import tensorflow as tf

params = model.trainable_variables

# numero totale parametri
d = np.sum([tf.size(p).numpy() for p in params])

F = np.zeros((d, d))

N = len(X_train_sv)

for x, y in zip(X_train_sv, y_train_sv):

    x = tf.convert_to_tensor([x], dtype=tf.float32)
    y = tf.convert_to_tensor([y])

    with tf.GradientTape() as tape:

        preds = model(x, training=False)

        loss = tf.keras.losses.sparse_categorical_crossentropy(
            y,
            preds
        )

    grads = tape.gradient(loss, params)

    # flatten gradienti
    g = np.concatenate([
        tf.reshape(grad, [-1]).numpy()
        for grad in grads
    ])

    # outer product
    F += np.outer(g, g)

# media
F /= N